------
# Data Loading
------

## Set Up
----

In [353]:
import numpy as np
import pandas as pd
import re
import email
from email.parser import Parser

## Data Loading
----

In [354]:
raw_emails_df  = pd.read_csv('../../data/emails.csv')

In [356]:
# Looking into format of message
raw_emails_df['message'][3]

"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Randall L Gay\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Dec2000\\Notes Folders\\'sent mail\nX-Origin: Allen-P\nX-FileName: pallen.nsf\n\nRandy,\n\n Can you send me a schedule of the salary and level of everyone in the \nscheduling group.  Plus your thoughts on any changes that need to be made.  \n(Patti S for example)\n\nPhillip"

## Feature Extraction
-----

Using `email` library to parse emails and extract relevant information from `message`: 

    - To
    - From
    - Subject
    - Content

Reference:
https://stackoverflow.com/questions/17872094/python-how-to-parse-things-such-as-from-to-body-from-a-raw-email-source-w

In [378]:
email_df = raw_emails_df.copy()

In [416]:
def extract_info(row):
    """
    Description: 
        Extract necessary info from emails: To, From, Subject, Body

    Input:
        Row of a dataframe

    Output:
        Dict of email information where keys are: To, From, Subject,Body
    """

    parser = Parser()
    my_email = parser.parsestr(row['message'])

    # Dict to hold email info
    email_info = {
        'from': my_email.get('From'),
        'to': my_email.get('To'),
        'subject': my_email.get('Subject'),
    }

    # Add email body to dict
    #   body of an email can be made up of attachments, text, and some HTML
    if my_email.is_multipart():
        # if email has multiple parts, body of the email can be in any one of these parts.
        body = ""
        # iterate over a list of parts in the email (multipart)
        for part in my_email.get_payload():
            # to extract content of each part
            body += part.get_payload()
        # using strip to remvoe whitespaces
        email_info['body'] = body.strip()
    else:
        email_info['body'] = my_email.get_payload().strip()

    return email_info

In [382]:
# apply function to df at row level (axis=1)
email_info = email_df.apply(extract_info, axis = 1)

In [411]:
# convert email_info to dataframe
email_data_df = pd.DataFrame(email_info.tolist())

In [412]:
email_data_df

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.
...,...,...,...,...
517396,john.zufferli@enron.com,kori.loibl@enron.com,Trade with John Lavorato,This is a trade with OIL-SPEC-HEDGE-NG (John L...
517397,john.zufferli@enron.com,john.lavorato@enron.com,Gas Hedges,Some of my position is with the Alberta Term b...
517398,john.zufferli@enron.com,dawn.doucet@enron.com,RE: CONFIDENTIAL,2\n\n -----Original Message-----\nFrom: \tDouc...
517399,john.zufferli@enron.com,jeanie.slone@enron.com,Calgary Analyst/Associate,Analyst\t\t\t\t\tRank\n\nStephane Brodeur\t\t\...


## Summary
----

So far, I have successfully loaded the email data from the CSV and have managed to extract key data such as to, from, subject and content using Email Parser.

Next step is to take a deeper look into the data, clean it and prepare for clustering.

## Appendix
------

Includes attempt of extracting email info using regular expressions. 

Ultimately, this method didn't work as the format of emails are not standard. For example, some emails contained multiple recipients and delimeters separating recipients are not concistent (for example: some include new line characters others do not). Some emails often contained whitespace, tabs or other characters which were inconcistent. Email Parser also worked better in extracting the content of emails since the body of emails seemed to be nested and simple pattern matching may miss important content.

### To:

In [418]:
def get_email_to(email):
    email_to = re.findall(r'\nTo:\s([a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})',email)
    
    if not email_to:
        return 'Missing Recipient'
    else:
        return email_to

In [419]:
get_email_to(raw_emails_df['message'][0])

['tim.belden@enron.com']

In [421]:
get_email_to(raw_emails_df['message'][65465])

['amozes@covad.com']

Fails to retrieve full list of email recipients

### From:

In [422]:
def get_email_from(email):
    email_from = re.findall(r'\nFrom:\s([a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})',email)
    
    if not email_from:
        return 'Missing Sender'
    else:
        return email_from

In [424]:
get_email_from(raw_emails_df['message'][45325])

['perfmgmt@enron.com']

### Subject:

In [425]:
def get_email_subject(email):
    email_from = re.findall(r'\nSubject:\s([^,|/n]+)',email)
    
    if not email_from:
        return 'Missing Subject'
    else:
        return email_from

In [427]:
get_email_subject(raw_emails_df['message'][0])

['\nMime-Versio']

Fails to notice empty subject, body of email is nested in MIME structure and so this should be ignored.

No point in continuing use of regexp, to find a better way to extract needed info.